In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, round as _round
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("DataEngineering_HW1").getOrCreate()

df_users = spark.read.csv("users.csv", header=True, inferSchema=True)
df_purchases = spark.read.csv("purchases.csv", header=True, inferSchema=True)
df_products = spark.read.csv("products.csv", header=True, inferSchema=True)

df_users_clean = df_users.dropna()
df_purchases_clean = df_purchases.dropna()
df_products_clean = df_products.dropna()

df_sales = df_purchases_clean.join(df_products_clean, "product_id") \
    .withColumn("total_amount", col("quantity") * col("price"))

total_by_category = df_sales.groupBy("category") \
    .agg(_sum("total_amount").alias("total_sales_all_ages"))
print("Загальна сума покупок за категоріями:")
total_by_category.show()

df_full = df_sales.join(df_users_clean, "user_id")

df_young_adults = df_full.filter((col("age") >= 18) & (col("age") <= 25))

category_sales_18_25 = df_young_adults.groupBy("category") \
    .agg(_sum("total_amount").alias("category_total_18_25"))
print("Сума покупок за категоріями (вік 18-25):")
category_sales_18_25.show()

window_spec = Window.partitionBy()
df_share = category_sales_18_25.withColumn(
    "grand_total_18_25", _sum("category_total_18_25").over(window_spec)
).withColumn(
    "percentage", _round((col("category_total_18_25") / col("grand_total_18_25")) * 100, 2)
).drop("grand_total_18_25")
print("Частка покупок за категоріями у відсотках (вік 18-25):")
df_share.show()

top_3_categories = df_share.orderBy(col("percentage").desc()).limit(3)
print("Топ-3 категорії за часткою витрат (вік 18-25):")
top_3_categories.show()

Загальна сума покупок за категоріями:
+-----------+--------------------+
|   category|total_sales_all_ages|
+-----------+--------------------+
|       Home|  1523.4999999999998|
|     Sports|  1802.4999999999998|
|Electronics|  1174.7999999999997|
|   Clothing|               790.3|
|     Beauty|   459.8999999999999|
+-----------+--------------------+

Сума покупок за категоріями (вік 18-25):
+-----------+--------------------+
|   category|category_total_18_25|
+-----------+--------------------+
|       Home|               361.1|
|     Sports|  310.49999999999994|
|Electronics|               249.6|
|   Clothing|               245.0|
|     Beauty|  41.400000000000006|
+-----------+--------------------+

Частка покупок за категоріями у відсотках (вік 18-25):
+-----------+--------------------+----------+
|   category|category_total_18_25|percentage|
+-----------+--------------------+----------+
|       Home|               361.1|      29.9|
|     Sports|  310.49999999999994|     25.71|
|Ele